# parents-dict-by-argidx — worked example 3: All-scalar args produces empty parents dict

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `parents-dict-by-argidx`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

When a forward function is called with no Tensor arguments at all — only scalars, shape tuples, or None — the parents dict is empty `{}`. This is a valid state: it means the output has no inputs to backpropagate through. Downstream code handles this gracefully because it loops over `parents.items()`, which simply yields nothing for an empty dict.

## Worked solution

**Step 1 — Enumerate all args.** We check each argument: `3.14` is a float, `(2, 3)` is a tuple, `None` is NoneType, `7` is an int — none are MiniTensor.

**Step 2 — Filter yields nothing.** The comprehension `{idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)}` evaluates to `{}` because no element passes the `isinstance` test.

**Step 3 — Verify the empty result.** An empty dict is returned. This is correct behaviour — the op still ran, but there are no parent Tensors to propagate gradients into.

**Step 4 — Demonstrate safe downstream use.** Iterating over `{}.items()` produces zero iterations, so any backward loop that calls `for argnum, parent in parents.items()` simply does nothing — safe and correct.

In [ ]:
import torch as t
from dataclasses import dataclass
from typing import Any

@dataclass
class MiniTensor:
    array: Any
    grad: Any = None

def build_parents_preserve_idx(args: tuple) -> dict:
    return {idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)}

# All-scalar scenario
args_all_scalar = (3.14, (2, 3), None, 7)
parents_empty = build_parents_preserve_idx(args_all_scalar)

print(f'Parents: {parents_empty}')          # {}
print(f'Is empty: {len(parents_empty) == 0}')  # True

# Demonstrate: backward loop is safely a no-op
back_fns_called = []
for argnum, parent in parents_empty.items():
    back_fns_called.append(argnum)
print(f'Back fns called: {back_fns_called}')  # []
print('Loop over empty parents is safe.')     # confirmed